In [1]:
!pip install -q transformers peft accelerate bitsandbytes datasets trl safetensors pandas lxml


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/60.7 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.3/60.7 MB 9.8 MB/s eta 0:00:07

   ━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/60.7 MB 86.7 MB/s eta 0:00:01

   ━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/60.7 MB 217.8 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━ 23.9/60.7 MB 217.9 MB/s eta 0:00:01

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 60.7/60.7 MB 259.8 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 60.7/60.7 MB 259.8 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 60.7/60.7 MB 259.8 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 60.7/60.7 MB 259.8 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 41.1 MB/s eta 0:00:00


In [2]:
import os, re, csv, time, random
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


Torch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4


In [3]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

CONFIG = {
    'model_name': 'Qwen/Qwen2.5-Coder-1.5B-Instruct',
    # ✅ FIX 1: 512→768 — many SVGs exceed 512 tokens and get silently truncated
    'max_seq_length': 768,
    # ✅ FIX 2: r 8→16 — more LoRA capacity, only ~10% more training time
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'learning_rate': 1e-4,
    # ✅ FIX 3: epochs 1→2 — model barely converges in 1 epoch on SVG structure
    'num_train_epochs': 2,
    'per_device_train_batch_size': 1,
    # ✅ FIX 4: accum 2→4 — effective batch=4, more stable gradient estimates
    'gradient_accumulation_steps': 4,
    'warmup_ratio': 0.05,
    'weight_decay': 0.01,
    'logging_steps': 20,
    'save_steps': 1000,
    # ✅ FIX 5: 2000→4000 samples — better diversity, fits in Kaggle time budget
    'max_train_samples': 4000,
    # ✅ FIX 6: 2500→3500 — recover more usable training SVGs
    'svg_max_len': 3500,
    'output_dir': '/kaggle/working/fast',
}
CONFIG


{'model_name': 'Qwen/Qwen2.5-Coder-1.5B-Instruct',
 'max_seq_length': 768,
 'lora_r': 16,
 'lora_alpha': 32,
 'lora_dropout': 0.05,
 'learning_rate': 0.0001,
 'num_train_epochs': 2,
 'per_device_train_batch_size': 1,
 'gradient_accumulation_steps': 4,
 'warmup_ratio': 0.05,
 'weight_decay': 0.01,
 'logging_steps': 20,
 'save_steps': 1000,
 'max_train_samples': 4000,
 'svg_max_len': 3500,
 'output_dir': '/kaggle/working/fast'}

## 1. Load Data

In [4]:
TRAIN_PATH = '/kaggle/input/datasets/rebeccachenn/dlmidterm/train.csv'
TEST_PATH  = '/kaggle/input/datasets/rebeccachenn/dlmidterm/test.csv'

train_df = pd.read_csv(TRAIN_PATH, engine='python', quotechar='"',
                        quoting=csv.QUOTE_ALL, on_bad_lines='skip')
test_df  = pd.read_csv(TEST_PATH,  engine='python', quoting=csv.QUOTE_MINIMAL)

print('train columns:', train_df.columns.tolist())
print('train shape  :', train_df.shape)
print('test shape   :', test_df.shape)
train_df.head(2)


train columns: ['id', 'prompt', 'svg']
train shape  : (50000, 3)
test shape   : (1000, 2)


,id,prompt,svg
0,fd61e324e0cec5c11f337d0bfe2858c8,The image features two orange squares with a m...,"<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
1,999b3d4d5a860725bf9528910b5612f3,A simple smiley face with a wide open mouth an...,"<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."


## 2. Helper Functions

In [5]:
ALLOWED_TAGS = {
    'svg','g','path','rect','circle','ellipse','line','polyline','polygon',
    'defs','use','symbol','clipPath','mask','linearGradient','radialGradient',
    'stop','text','tspan','title','desc','style','pattern','marker','filter',
}
DISALLOWED_ATTRS_RE = re.compile(
    r'\s+(?:filling|onclick|onload|onmouseover|xmlns:xlink)=["\'][^"\']*("|\')',
    re.IGNORECASE
)

def normalize_svg(svg_text):
    if not svg_text: return ''
    svg_text = DISALLOWED_ATTRS_RE.sub('', svg_text)
    for attr in ('width', 'height', 'viewBox'):
        svg_text = re.sub(rf'(<svg\b[^>]*?)\b{attr}=["\'][^"\']*("|\')', r'\1', svg_text)
    svg_text = re.sub(
        r'(<svg\b)',
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"',
        svg_text, count=1
    )
    svg_text = re.sub(r'(xmlns="[^"]*")(?=.*xmlns="[^"]*")', '', svg_text)
    svg_text = re.sub(r'\s+', ' ', svg_text).strip()
    return svg_text

def has_disallowed_tags(svg_text):
    try:
        root = ET.fromstring(svg_text)
        for elem in root.iter():
            tag = elem.tag
            if isinstance(tag, str):
                local = tag.split('}')[-1] if '}' in tag else tag
                if local not in ALLOWED_TAGS: return True
        return False
    except Exception: return True

def is_valid_svg(svg_text):
    if not svg_text: return False
    try:
        root = ET.fromstring(svg_text)
        tag = root.tag.split('}')[-1] if '}' in root.tag else root.tag
        return tag == 'svg'
    except ET.ParseError: return False

print('Helper functions defined.')


Helper functions defined.


## 3. Data Subset Sampling

In [6]:
print(f'Raw train rows: {len(train_df)}')

# Step 1: Fast string-level pre-filter (no XML parsing)
quick_mask = (
    train_df['prompt'].notna() &
    train_df['svg'].notna() &
    (train_df['prompt'].astype(str).str.strip() != '') &
    (train_df['svg'].astype(str).str.strip() != '') &
    (train_df['svg'].astype(str).str.lower().str.startswith('<svg')) &
    (train_df['svg'].astype(str).str.len() <= CONFIG['svg_max_len']) &
    (train_df['prompt'].astype(str).str.len() <= 300)
)
quick_pool = train_df[quick_mask].reset_index(drop=True)
print(f'Rows passing quick filter: {len(quick_pool)} / {len(train_df)}')

# Step 2: Length distribution
svg_lens    = quick_pool['svg'].astype(str).str.len()
prompt_lens = quick_pool['prompt'].astype(str).str.len()
print(f'SVG    — median:{svg_lens.median():.0f}  p90:{svg_lens.quantile(0.9):.0f}  p95:{svg_lens.quantile(0.95):.0f}')
print(f'Prompt — median:{prompt_lens.median():.0f}  p95:{prompt_lens.quantile(0.95):.0f}')

# Step 3: Sample 2× target for cleaning
POOL_SIZE = min(CONFIG['max_train_samples'] * 2, len(quick_pool))
sample_pool = quick_pool.sample(n=POOL_SIZE, random_state=SEED).reset_index(drop=True)
print(f'Cleaning pool size: {POOL_SIZE}')

# Step 4: Full XML clean on pool only
df = sample_pool[['id', 'prompt', 'svg']].copy()
df['prompt'] = df['prompt'].astype(str).str.strip()
df['svg']    = df['svg'].astype(str).str.strip()
df['svg']    = df['svg'].apply(normalize_svg)
df = df[df['svg'].apply(is_valid_svg)].reset_index(drop=True)
df = df[~df['svg'].apply(has_disallowed_tags)].reset_index(drop=True)
df = df.drop_duplicates(subset=['svg']).reset_index(drop=True)
print(f'Usable after cleaning: {len(df)}')

# Step 5: Final sample
train_small_df = df.sample(
    n=min(CONFIG['max_train_samples'], len(df)),
    random_state=SEED
).reset_index(drop=True)
print(f'Final training set: {len(train_small_df)}')
train_small_df.head(2)


Raw train rows: 50000


Rows passing quick filter: 36451 / 50000
SVG    — median:1568  p90:2995  p95:3231
Prompt — median:102  p95:216
Cleaning pool size: 8000


Usable after cleaning: 7991
Final training set: 4000


,id,prompt,svg
0,69f18567d2d66a040e3f14ba348e69cd,"The image consists of a black, abstract shape ...","<svg width=""256"" height=""256"" viewBox=""0 0 256..."
1,f5bfdefa-048b-46c6-b3c1-e7ae9af8a8c1,A black and white line drawing of an oak leaf.,"<svg width=""256"" height=""256"" viewBox=""0 0 256..."


## 4. System Prompt

In [7]:
SYSTEM_PROMPT = (
    'You are an SVG code generator. '
    'Given a text description, output ONLY valid SVG code with '
    'width="256" height="256" viewBox="0 0 256 256". '
    'Do not include explanations or markdown code fences.'
)
print('System prompt set.')


System prompt set.


## 5. Load Model & LoRA

In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'], use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    CONFIG['model_name'],
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
print('Loaded:', CONFIG['model_name'])


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-Coder-1.5B-Instruct


In [9]:
lora_config = LoraConfig(
    r=CONFIG['lora_r'], lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none', task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## 6. Build Dataset

In [10]:
# ✅ RESTORED: apply_chat_template ensures train/inference format is identical
# The previous 'Prompt: ... SVG: ...' format doesn't use Qwen's chat tokens,
# so the model can't distinguish system/user/assistant roles during inference.
def make_chat_text(row):
    messages = [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': row['prompt']},
        {'role': 'assistant', 'content': row['svg']},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

train_small_df['text'] = train_small_df.apply(make_chat_text, axis=1)
print(train_small_df['text'].iloc[0][:600])
print(f'Avg text length: {train_small_df["text"].str.len().mean():.0f} chars')


<|im_start|>system
You are an SVG code generator. Given a text description, output ONLY valid SVG code with width="256" height="256" viewBox="0 0 256 256". Do not include explanations or markdown code fences.<|im_end|>
<|im_start|>user
The image consists of a black, abstract shape resembling a stylized letter 'G' or a curved, wavy form with a jagged bottom edge against a white background.<|im_end|>
<|im_start|>assistant
<svg width="256" height="256" viewBox="0 0 256 256" xmlns="http://www.w3.org/2000/svg" ><path fill="" fill-opacity="1.0" d="M95.77558898925781 190.6183624267578 C116.3923873901
Avg text length: 2011 chars


In [11]:
sample_texts = train_small_df['text'].sample(min(200, len(train_small_df)), random_state=SEED).tolist()
token_lengths = [len(tokenizer(t, truncation=False)['input_ids']) for t in sample_texts]
truncated = sum(1 for l in token_lengths if l > CONFIG['max_seq_length'])
print(f'Median tokens : {int(np.median(token_lengths))}')
print(f'95th pct      : {int(np.percentile(token_lengths, 95))}')
print(f'Max tokens    : {max(token_lengths)}')
print(f'Truncated (>{CONFIG["max_seq_length"]}): {truncated}/{len(sample_texts)} ({truncated/len(sample_texts)*100:.1f}%)')


Median tokens : 1496
95th pct      : 3143
Max tokens    : 3381
Truncated (>768): 156/200 (78.0%)


In [12]:
train_dataset = Dataset.from_pandas(train_small_df[['text']], preserve_index=False)
print(f'Dataset size: {len(train_dataset)}')


Dataset size: 4000


## 7. Training

In [13]:
sft_config = SFTConfig(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['num_train_epochs'],
    per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    warmup_ratio=CONFIG['warmup_ratio'],
    weight_decay=CONFIG['weight_decay'],
    bf16=True,     
    logging_steps=CONFIG['logging_steps'],
    save_strategy='steps',
    save_steps=CONFIG['save_steps'],
    max_grad_norm=0.3,
    lr_scheduler_type='cosine',
    optim='paged_adamw_8bit',
    report_to='none',
    seed=SEED,
    remove_unused_columns=False,
    max_length=CONFIG['max_seq_length'],
    dataset_text_field='text',
    packing=False,
)
trainer = SFTTrainer(
    model=model, args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

In [14]:
train_result = trainer.train()
print(train_result)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
20,1.070461
40,0.848101
60,0.668882
80,0.507126
100,0.507562
120,0.559913
140,0.472127
160,0.437341
180,0.535565
200,0.490607


TrainOutput(global_step=2000, training_loss=0.6029628648757934, metrics={'train_runtime': 32272.2047, 'train_samples_per_second': 0.248, 'train_steps_per_second': 0.062, 'total_flos': 4.429193745602765e+16, 'train_loss': 0.6029628648757934})


In [15]:
os.makedirs(CONFIG['output_dir'], exist_ok=True)
trainer.save_model(CONFIG['output_dir'])
tokenizer.save_pretrained(CONFIG['output_dir'])
print(f'Saved to: {CONFIG["output_dir"]}')


Saved to: /kaggle/working/fast


## 8. Inference

In [16]:
# Re-enable KV cache after training
model.eval()
model.config.use_cache = True
if hasattr(model, 'gradient_checkpointing_disable'):
    model.gradient_checkpointing_disable()
print('Model in eval mode, cache enabled.')


Model in eval mode, cache enabled.


In [17]:
SVG_REGEX = re.compile(r'(<svg\b[\s\S]*?</svg>)', flags=re.IGNORECASE)

def extract_svg(text):
    if not text: return ''
    for tok in ['<|im_end|>', '<|endoftext|>']:
        idx = text.find(tok)
        if idx != -1: text = text[:idx]
    match = SVG_REGEX.search(text)
    if match: return match.group(1).strip()
    start = text.lower().find('<svg')
    if start != -1:
        partial = text[start:].strip()
        if not partial.lower().endswith('</svg>'): partial += '</svg>'
        return partial
    return ''

def fallback_svg():
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="#cccccc"/>'
        '</svg>'
    )

def generate_svg(prompt, max_new_tokens=500):
    # ✅ FIX: use apply_chat_template — must match training format exactly
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': prompt},   # ✅ FIX: no [:120] truncation
    ]
    chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(chat_text, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,   # ✅ NEW: reduces repeating path segments
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    decoded = tokenizer.decode(output_ids[0][input_len:], skip_special_tokens=False)
    svg = normalize_svg(extract_svg(decoded))
    used_fallback = not is_valid_svg(svg) or has_disallowed_tags(svg) or len(svg) > 15900
    if used_fallback: svg = fallback_svg()
    return svg, decoded, used_fallback

print('Inference functions defined.')


Inference functions defined.


In [18]:
for p in ['a simple blue bird icon',
          'a curved arrow in a square',
          'five horizontal lines of varying thickness']:
    svg, _, fb = generate_svg(p, max_new_tokens=500)
    print(f'[fallback={fb}] valid={is_valid_svg(svg)} len={len(svg)}')
    print(f'  {svg[:300]}')
    print('-'*60)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[fallback=False] valid=True len=527
  <svg width="256" height="256" viewBox="0 0 256 256" xmlns="http://www.w3.org/2000/svg" ><path fill="#1C4790" fill-opacity="1.0" d="M18.75 12.5 L18.75 18.75 C18.75 22.5 22.5 22.5 25.0 22.5 L175.0 22.5 C177.5 22.5 181.25 22.5 181.25 18.75 L181.25 12.5 L18.75 12.5 Z M18.75 18.75 L18.75 18.75 L18.75 12.
------------------------------------------------------------


[fallback=True] valid=True len=199
  <svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><rect x="0" y="0" width="256" height="256" fill="white"/><circle cx="128" cy="128" r="64" fill="#cccccc"/></svg>
------------------------------------------------------------


[fallback=False] valid=True len=137
  <svg width="256" height="256" viewBox="0 0 256 256" xmlns="http://www.w3.org/2000/svg"><path d="m14 18v-7h-12v7zm0-.01h-12v7h12z"/></svg>
------------------------------------------------------------


## 9. Generate Submissions

In [19]:
results, fallback_count = [], 0

for i, row in test_df.iterrows():
    svg, _, fb = generate_svg(row['prompt'], max_new_tokens=500)
    if fb: fallback_count += 1
    results.append({'id': row['id'], 'svg': svg})
    if (i + 1) % 50 == 0:
        print(f'Processed {i+1}/{len(test_df)} | fallback so far: {fallback_count}')

submission_df = pd.DataFrame(results)
print(f'Total:{len(submission_df)} | Fallback:{fallback_count} ({fallback_count/len(submission_df)*100:.1f}%)')


Processed 50/1000 | fallback so far: 28


Processed 100/1000 | fallback so far: 61


Processed 150/1000 | fallback so far: 87


In [ ]:
valid_count = submission_df['svg'].apply(is_valid_svg).sum()
print(f'Valid          : {valid_count}/{len(submission_df)} ({valid_count/len(submission_df)*100:.1f}%)')
print(f'Disallowed tags: {submission_df["svg"].apply(has_disallowed_tags).sum()}')
print(f'Over 16000 chars: {(submission_df["svg"].str.len() > 16000).sum()}')


In [ ]:
submission_df.to_csv('/kaggle/working/submission.csv', index=False)
print('Saved: /kaggle/working/submission.csv')
submission_df.head()
